# Notebook 05 — Multi-head attention

Paired with `theory/05_multi_head.md`. **Mode:** mixed.

## QHMPC

**Question.** Verify the structural claims about MHA and run one short training experiment to probe head specialization:
1. Parameter and FLOP counts: MHA vs. single-head at fixed $d_{\text{model}}$.
2. Rank of pre- and post-softmax attention matrices (Exercise 5.3).
3. Head specialization on a synthetic retrieve-from-context task.
4. Head importance via ablation.
5. Minimum head count for task performance.

**Hypothesis.** Props 5.2, 5.3, 5.6 are exact. Empirical obs 5.4 and 5.5 appear in the training experiment if the task has enough structure.

**Method.** Experiments 1–2: structural checks. Experiments 3–5: train a tiny transformer, ablate.

**Prediction.** *Paste §5.7 predictions here.*

**Confidence.** Overall: *[low / medium / high, reason]*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)

---
## Experiment 1 — parameter and FLOP counts

Confirm Prop 5.3: total FLOPs and parameter count are invariant under choice of $h$ at fixed $d_{\text{model}}$.

In [ ]:
class MHA(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.d_model, self.h, self.d_h = d_model, h, d_model // h
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X, return_attn=False):
        B, n, d = X.shape
        Q = self.W_Q(X).view(B, n, self.h, self.d_h).transpose(1, 2)  # (B, h, n, d_h)
        K = self.W_K(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        V = self.W_V(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        scores = Q @ K.transpose(-1, -2) / (self.d_h ** 0.5)
        A = scores.softmax(-1)                                        # (B, h, n, n)
        out = A @ V                                                    # (B, h, n, d_h)
        out = out.transpose(1, 2).contiguous().view(B, n, d)
        out = self.W_O(out)
        return (out, A) if return_attn else out

d_model = 128
print(f'{"h":>4}  {"d_h":>4}  {"params":>10}')
for h in [1, 2, 4, 8, 16]:
    m = MHA(d_model, h)
    params = sum(p.numel() for p in m.parameters())
    print(f'{h:>4}  {d_model // h:>4}  {params:>10}')

print(f'\nExpected: 4 * d_model^2 = {4 * d_model ** 2}  (invariant in h)')

**Sub-finding 1.** *Parameter count invariant under $h$? Expected by Prop 5.3.*

---
## Experiment 2 — rank of pre- and post-softmax attention (Exercise 5.3)

For $h = 4, d_h = 8, n = 32$: predict $\text{rank}(S) \le d_h = 8$, but $\text{rank}(A) = n = 32$ almost surely.

In [ ]:
n, d_h = 32, 8
Q = np.random.randn(n, d_h)
K = np.random.randn(n, d_h)
S = Q @ K.T / np.sqrt(d_h)
A = np.exp(S - S.max(axis=-1, keepdims=True))
A = A / A.sum(axis=-1, keepdims=True)

rank_S = np.linalg.matrix_rank(S, tol=1e-8)
rank_A = np.linalg.matrix_rank(A, tol=1e-8)
print(f'rank(S) = {rank_S}   (predicted <= d_h = {d_h})')
print(f'rank(A) = {rank_A}   (predicted = n = {n})')

s_eigs = np.linalg.svd(S, compute_uv=False)
a_eigs = np.linalg.svd(A, compute_uv=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].semilogy(s_eigs, 'o-'); axes[0].set_title('Singular values of S (pre-softmax)')
axes[0].set_xlabel('index'); axes[0].set_ylabel('sigma')
axes[1].semilogy(a_eigs, 'o-'); axes[1].set_title('Singular values of A (post-softmax)')
axes[1].set_xlabel('index')
plt.tight_layout(); plt.show()

**Sub-finding 2.** *Did softmax lift rank from $d_h$ to $n$? What nonlinearity in softmax does this — the exponential? What would a linear softmax replacement (softmax-attention without the exp, i.e., $\alpha = s / \sum s$ with ReLU) do to the rank?*

---
## Experiment 3 — head specialization on a synthetic retrieve task

Train a 1-layer $h = 4$ transformer on a toy retrieve-from-context task. Sequence: `[k1, v1, k2, v2, ..., k5, v5, query]`. Model must output the value associated with the matching key.

Keep it small — should train in ~30 seconds on CPU.

In [ ]:
# Task: vocabulary of 16 'key' symbols and 16 'value' symbols. Sequence length 11 (5 kv-pairs + 1 query).
V_key, V_val = 16, 16
seq_len = 11
d_model = 32
n_pairs = 5

def make_batch(batch_size):
    keys = torch.randint(0, V_key, (batch_size, n_pairs))
    vals = torch.randint(0, V_val, (batch_size, n_pairs))
    query_idx = torch.randint(0, n_pairs, (batch_size,))
    targets = vals[torch.arange(batch_size), query_idx]
    # Interleave: [k1, v1, k2, v2, ..., k5, v5, query]. Encode keys as 0..15, values as 16..31, query as its key id.
    seq = torch.zeros(batch_size, seq_len, dtype=torch.long)
    for b in range(batch_size):
        for p in range(n_pairs):
            seq[b, 2*p] = keys[b, p]
            seq[b, 2*p + 1] = V_key + vals[b, p]
        seq[b, -1] = keys[b, query_idx[b]]
    return seq, targets

V_total = V_key + V_val  # 32

class Model(nn.Module):
    def __init__(self, h):
        super().__init__()
        self.embed = nn.Embedding(V_total, d_model)
        self.pos = nn.Parameter(torch.randn(seq_len, d_model) * 0.02)
        self.mha = MHA(d_model, h)
        self.out = nn.Linear(d_model, V_val, bias=False)

    def forward(self, seq, return_attn=False):
        x = self.embed(seq) + self.pos
        if return_attn:
            y, A = self.mha(x, return_attn=True)
        else:
            y = self.mha(x)
        logits = self.out(y[:, -1])  # predict from the query position
        return (logits, A) if return_attn else logits

torch.manual_seed(0)
model = Model(h=4)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)

loss_history = []
for step in range(2000):
    seq, targets = make_batch(64)
    logits = model(seq)
    loss = F.cross_entropy(logits, targets)
    opt.zero_grad(); loss.backward(); opt.step()
    loss_history.append(loss.item())

plt.figure(figsize=(6, 3))
plt.plot(loss_history); plt.xlabel('step'); plt.ylabel('loss'); plt.yscale('log')
plt.title('Training loss — retrieve task')
plt.show()

# Final accuracy
with torch.no_grad():
    seq, targets = make_batch(1000)
    acc = (model(seq).argmax(-1) == targets).float().mean().item()
    print(f'Final accuracy: {acc:.3f}   (chance = 1/{V_val} = {1/V_val:.3f})')

## Experiment 4 — visualize per-head attention after training

On a single batch, plot the $4$ heads' attention matrices side by side. Do they look the same, or different?

In [ ]:
model.eval()
seq, targets = make_batch(1)
with torch.no_grad():
    _, A = model(seq, return_attn=True)
A = A[0].numpy()  # (h, n, n)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for i in range(4):
    im = axes[i].imshow(A[i], cmap='viridis', vmin=0, vmax=A.max())
    axes[i].set_title(f'Head {i}')
    axes[i].set_xlabel('key pos'); axes[i].set_ylabel('query pos')
plt.colorbar(im, ax=axes[-1])
plt.tight_layout(); plt.show()

# Query row (last position): where does each head look?
print('Query-row attention, per head (last row of each matrix):')
for i in range(4):
    print(f'  head {i}: {A[i, -1]}')

**Sub-finding 4.** *Did heads specialize to different patterns, or do they all look the same? For the query row (last position), which head attends to the correct key position?*

---
## Experiment 5 — head ablation

Zero out each head independently; measure the resulting accuracy. Heads whose ablation hurts most are the "important" ones.

In [ ]:
def eval_with_head_mask(model, head_mask):
    """head_mask: tensor of shape (h,), 1=keep, 0=ablate."""
    def patched_mha(self, X, return_attn=False):
        B, n, d = X.shape
        Q = self.W_Q(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        K = self.W_K(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        V = self.W_V(X).view(B, n, self.h, self.d_h).transpose(1, 2)
        scores = Q @ K.transpose(-1, -2) / (self.d_h ** 0.5)
        A = scores.softmax(-1)
        out = A @ V
        out = out * head_mask.view(1, -1, 1, 1)           # zero out ablated heads
        out = out.transpose(1, 2).contiguous().view(B, n, d)
        return self.W_O(out)

    import types
    model.mha.forward = types.MethodType(patched_mha, model.mha)
    with torch.no_grad():
        seq, targets = make_batch(1000)
        acc = (model(seq).argmax(-1) == targets).float().mean().item()
    # restore
    model.mha.forward = types.MethodType(MHA.forward, model.mha)
    return acc

# Baseline
with torch.no_grad():
    seq, targets = make_batch(1000)
    baseline = (model(seq).argmax(-1) == targets).float().mean().item()

print(f'Baseline (all heads): {baseline:.3f}\n')
print(f'{"Ablated":>10}  {"Accuracy":>10}  {"Δ from baseline":>16}')
print('-' * 42)
for i in range(4):
    mask = torch.ones(4); mask[i] = 0
    acc = eval_with_head_mask(model, mask)
    print(f'head {i}:     {acc:.3f}         {acc - baseline:+.3f}')

**Sub-finding 5.** *Which head was most important by ablation? Was it the one with the most targeted query-row attention in Experiment 4? Is one head essentially doing all the work, or is the task distributed?*

---
## Finding

*3–6 sentences. Did head specialization emerge? Was the task solvable by one head alone after training? What does the ablation say about head redundancy vs. head necessity in this toy setup?*